# Skill-Scoring Dataset — Short EDA

Quick look at the synthetic corpus that feeds the scoring model:

1. **Data variance** — label balance, structural spread, document lengths
2. **Flags summary** — allocation rationality, skill-showcase deltas, shortcut checks
3. **Persona-level skill flagging** — drop a `(persona, skill)` row when, across that
   persona's documents, `max |delta| >= 3` **or** `avg |delta| >= 2`
4. **Other findings**

Training unit = one `(persona, skill)` row (label = the persona's global skill level).
Run top-to-bottom with the project `.venv` kernel from the repo root.

In [16]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd
pd.set_option('display.max_rows', 60)

# display() is provided by the IPython kernel; define a fallback so the name
# resolves (and the notebook still runs) outside a notebook context too.
try:
    from IPython.display import display
except ImportError:
    def display(*objs):
        for o in objs:
            print(o)

DATA = Path('data')
personas = json.load(open(DATA / 'personas.json', encoding='utf-8'))
db = json.load(open(DATA / 'documents_db.json', encoding='utf-8'))

def load_report(name):
    p = DATA / 'reports' / name
    return json.load(open(p, encoding='utf-8')) if p.exists() else None

showcase  = load_report('skill_showcase_report.json')
alloc_rep = load_report('allocation_rationality_report.json')
shortcut  = load_report('shortcut_check_report.json')

def norm(s):
    return s.strip().lower().replace(' ', '_').replace('/', '_').replace('-', '_')

doc_persona = {d['doc_id']: d['persona_id'] for d in db.values()}
docs_by_persona = defaultdict(list)
for d in db.values():
    docs_by_persona[d['persona_id']].append(d)

def dist(values):
    s = pd.Series(values).value_counts(dropna=False)
    out = s.rename('count').to_frame()
    out['pct'] = (100 * out['count'] / out['count'].sum()).round(1)
    return out

print(f"{len(personas)} personas | {len(db)} documents")

310 personas | 1172 documents


## 1. Data variance

In [17]:
# Training rows = one (persona, skill) pair
rows = [(p['persona_id'], sk, lv) for p in personas for sk, lv in p['skills'].items()]
rows_df = pd.DataFrame(rows, columns=['persona_id', 'skill', 'label'])
rows_df['nskill'] = rows_df['skill'].map(norm)
print(f"Training rows (persona x skill): {len(rows_df)}")

# Label / proficiency distribution -- the class balance the scorer must learn
label_dist = rows_df['label'].value_counts().sort_index().rename('count').to_frame()
label_dist['pct'] = (100 * label_dist['count'] / len(rows_df)).round(1)
label_dist

Training rows (persona x skill): 3444


,count,pct
label,,
1,372,10.8
2,571,16.6
3,971,28.2
4,997,28.9
5,533,15.5


In [18]:
# Structural spread: skills per persona, documents per persona
skills_per = pd.Series([len(p['skills']) for p in personas], name='skills_per_persona')
docs_per = pd.Series([len(docs_by_persona[p['persona_id']]) for p in personas], name='docs_per_persona')
pd.concat([skills_per.describe(), docs_per.describe()], axis=1).round(2)

,skills_per_persona,docs_per_persona
count,310.00,310.00
mean,11.11,3.78
std,2.08,1.44
min,8.00,1.00
25%,10.00,3.00
50%,11.00,4.00
75%,12.00,5.00
max,18.00,7.00


In [19]:
# Categorical variance: doc types, archetypes, seniority
print('--- document types ---');  display(dist([d['type'] for d in db.values()]))
print('--- archetypes ---');      display(dist([p['hyperparams'].get('archetype') for p in personas]))
print('--- seniority ---');       display(dist([p['hyperparams'].get('seniority') for p in personas]))

--- document types ---


,count,pct
project_readme,425,36.3
cv,310,26.5
recommendation,237,20.2
blog,140,11.9
linkedin,60,5.1


--- archetypes ---


,count,pct
backend_engineer,31,10.0
data_engineer,29,9.4
ml_engineer,26,8.4
data_scientist,24,7.7
tech_lead_architect,20,6.5
qa_test_engineer,20,6.5
devops_sre,19,6.1
platform_engineer,19,6.1
game_developer,19,6.1
mobile_developer,19,6.1


--- seniority ---


,count,pct
senior,94,30.3
junior,86,27.7
mid,68,21.9
staff,62,20.0


In [20]:
# Document length variance (words) by type
len_df = pd.DataFrame([{'type': d['type'], 'words': len(d.get('text', '').split())}
                       for d in db.values()])
len_df.groupby('type')['words'].describe()[['count', 'mean', 'std', 'min', 'max']].round(0)

,count,mean,std,min,max
type,,,,,
blog,140.0,375.0,52.0,258.0,552.0
cv,310.0,473.0,79.0,271.0,705.0
linkedin,60.0,173.0,23.0,122.0,220.0
project_readme,425.0,293.0,45.0,24.0,475.0
recommendation,237.0,219.0,28.0,118.0,287.0


In [21]:
# Does proficiency vary by role? (mean label per archetype)
arch_of = {p['persona_id']: p['hyperparams'].get('archetype') for p in personas}
rows_df['archetype'] = rows_df['persona_id'].map(arch_of)
rows_df.groupby('archetype')['label'].agg(['mean', 'std', 'count']).round(2).sort_values('mean')

,mean,std,count
archetype,,,
fullstack_developer,2.87,0.95,211
mobile_developer,2.93,1.22,192
game_developer,3.05,1.19,194
embedded_engineer,3.07,1.20,137
qa_test_engineer,3.15,1.24,204
security_engineer,3.16,1.28,192
frontend_engineer,3.16,1.29,180
platform_engineer,3.21,1.17,233
product_manager,3.22,1.21,168


## 2. Flags summary

In [22]:
# Allocation rationality -- whole-persona evidence coherence (LLM judged)
flagged_personas = [r['persona_id'] for r in alloc_rep['results'] if r.get('flagged')]
print(f"Allocation rationality: {len(flagged_personas)}/{alloc_rep['total_personas']} personas flagged "
      f"({100 * len(flagged_personas) / alloc_rep['total_personas']:.1f}%)")
print('rationality score distribution (1-5):')
dist([r['score'] for r in alloc_rep['results'] if 'score' in r]).sort_index()

Allocation rationality: 7/310 personas flagged (2.3%)
rationality score distribution (1-5):


,count,pct
2,7,2.3
3,68,21.9
4,116,37.4
5,119,38.4


In [23]:
# Skill-showcase comparisons (per doc x skill): allocated vs LLM-judged level
cmp = []
for r in showcase['results']:
    pid = doc_persona.get(r['doc_id'])
    for c in r['comparisons']:
        cmp.append({'persona_id': pid, 'doc_id': r['doc_id'], 'skill': norm(c['skill']),
                    'allocated': c['allocated_level'], 'judged': c['llm_judged_level'],
                    'abs_delta': abs(c['delta'])})
cmp_df = pd.DataFrame(cmp)
print(f"Total doc x skill comparisons: {len(cmp_df)}")

ad = cmp_df['abs_delta'].value_counts().sort_index().rename('count').to_frame()
ad['pct'] = (100 * ad['count'] / len(cmp_df)).round(1)
print('abs-delta distribution:'); display(ad)

# Flag rate at candidate thresholds
thr = pd.DataFrame([{'rule': f'|delta|>={t}',
                     'flagged': int((cmp_df['abs_delta'] >= t).sum()),
                     'pct': round(100 * (cmp_df['abs_delta'] >= t).mean(), 1)}
                    for t in (2, 3, 4)])
print('flag rate at thresholds (comparison grain):'); display(thr)

Total doc x skill comparisons: 9089
abs-delta distribution:


,count,pct
abs_delta,,
0,3390,37.3
1,4387,48.3
2,1053,11.6
3,191,2.1
4,68,0.7


flag rate at thresholds (comparison grain):


,rule,flagged,pct
0,|delta|>=2,1312,14.4
1,|delta|>=3,259,2.8
2,|delta|>=4,68,0.7


In [24]:
# Where do the disputes concentrate? Flag rate by allocated level
g = cmp_df.assign(f2=cmp_df['abs_delta'] >= 2, f3=cmp_df['abs_delta'] >= 3)
by_lv = g.groupby('allocated').agg(n=('abs_delta', 'size'), flag_ge2=('f2', 'sum'), flag_ge3=('f3', 'sum'))
by_lv['pct_ge2'] = (100 * by_lv['flag_ge2'] / by_lv['n']).round(1)
by_lv['pct_ge3'] = (100 * by_lv['flag_ge3'] / by_lv['n']).round(1)
by_lv

,n,flag_ge2,flag_ge3,pct_ge2,pct_ge3
allocated,,,,,
2,1719,128,7,7.4,0.4
3,3472,581,0,16.7,0.0
4,2622,214,145,8.2,5.5
5,1276,389,107,30.5,8.4


In [25]:
# Non-LLM shortcut checks
bp = shortcut['banned_phrases']
print(f"Banned phrases: {bp['documents_with_issues']}/{bp['total_documents']} docs flagged "
      f"({bp['total_issues']} issues)")
print(f"Level-1 absence violations: {shortcut['level1_absence']['violations']}")
bb = shortcut['bow_baseline']
print(f"Bag-of-words baseline CV accuracy: {bb['mean_cv_accuracy']:.3f} "
      f"(threshold {bb['threshold']}, passed={bb['passed']})")
print("  low BoW accuracy => labels are NOT trivially separable by keywords (good)")

Banned phrases: 3/1172 docs flagged (3 issues)
Level-1 absence violations: 11
Bag-of-words baseline CV accuracy: 0.325 (threshold 0.45, passed=True)
  low BoW accuracy => labels are NOT trivially separable by keywords (good)


## 3. Persona-level skill flagging

Aggregate the per-document `abs_delta` for each skill **up to the persona**, then drop the
`(persona, skill)` training row when `max |delta| >= 3` **or** `avg |delta| >= 2`.
The `max` clause catches a single egregious mis-showcase; the `avg` clause catches a skill
that is mildly-but-consistently off across documents.

In [26]:
# max / avg abs-delta per (persona, skill)
agg = (cmp_df.groupby(['persona_id', 'skill'])['abs_delta']
       .agg(max_abs='max', avg_abs='mean', n_docs='size').reset_index())
agg['flagged'] = (agg['max_abs'] >= 3) | (agg['avg_abs'] >= 2)

# Join onto the real training rows (never-judged skills are kept)
merged = rows_df.merge(agg, left_on=['persona_id', 'nskill'], right_on=['persona_id', 'skill'],
                       how='left', suffixes=('', '_agg'))
merged['flagged'] = merged['flagged'].fillna(False)

n_drop = int(merged['flagged'].sum())
print(f"Rule: drop (persona, skill) if max|delta|>=3 OR avg|delta|>=2")
print(f"Dropped {n_drop}/{len(merged)} rows = {100 * n_drop / len(merged):.1f}%  |  "
      f"kept {len(merged) - n_drop}")

Rule: drop (persona, skill) if max|delta|>=3 OR avg|delta|>=2
Dropped 252/3444 rows = 7.3%  |  kept 3192


In [27]:
# Drop impact per class (label)
tbl = merged.groupby('label').agg(before=('flagged', 'size'), dropped=('flagged', 'sum'))
tbl['after'] = tbl['before'] - tbl['dropped']
tbl['dropped_pct'] = (100 * tbl['dropped'] / tbl['before']).round(1)
tbl[['before', 'after', 'dropped', 'dropped_pct']]

,before,after,dropped,dropped_pct
label,,,,
1,372,372,0,0.0
2,571,554,17,3.0
3,971,956,15,1.5
4,997,892,105,10.5
5,533,418,115,21.6


In [28]:
# Worst offenders (highest max/avg abs-delta)
agg.sort_values(['max_abs', 'avg_abs'], ascending=False).head(10).reset_index(drop=True)

,persona_id,skill,max_abs,avg_abs,n_docs,flagged
0,p_061,mentoring,4,4.000000,1,True
1,p_135,team_leadership,4,3.200000,5,True
2,p_011,python,4,3.000000,3,True
3,p_127,javascript,4,3.000000,6,True
4,p_260,responsive_design,4,3.000000,3,True
5,p_005,javascript,4,2.800000,5,True
6,p_135,technical_debt_management,4,2.800000,5,True
7,p_011,rest_api_design,4,2.666667,3,True
8,p_125,tensorflow,4,2.666667,3,True
9,p_150,nlp,4,2.666667,3,True


## 4. Other findings

In [29]:
# Data-quality note: skill_evidence double-lists each skill (case variants 'aws' vs 'AWS')
dup_docs = sum(1 for d in db.values()
               if len({k.lower() for k in d.get('skill_evidence', {})}) != len(d.get('skill_evidence', {})))
print(f"Documents whose skill_evidence has case-duplicate keys: {dup_docs}/{len(db)}")
print("skill_evidence uses normalized keys (e.g. 'ci_cd_pipelines') while persona.skills")
print("uses display keys (e.g. 'CI/CD pipelines') -- norm() bridges them. Only a couple of")
print("docs additionally store both case variants, a minor source-side inconsistency.")

Documents whose skill_evidence has case-duplicate keys: 0/1172
skill_evidence uses normalized keys (e.g. 'ci_cd_pipelines') while persona.skills
uses display keys (e.g. 'CI/CD pipelines') -- norm() bridges them. Only a couple of
docs additionally store both case variants, a minor source-side inconsistency.


In [30]:
# Shortcut risk: is document length a giveaway of skill level?
from scipy.stats import spearmanr
pairs = [(inten, len(d.get('text', '').split()))
         for d in db.values() for inten in d.get('skill_evidence', {}).values()]
pe = pd.DataFrame(pairs, columns=['intensity', 'words'])
rho, p = spearmanr(pe['intensity'], pe['words'])
print(f"Spearman(skill intensity, doc word-count) = {rho:.3f}  (p={p:.1e}, n={len(pe)})")
print("low-to-moderate => length is not a strong shortcut for proficiency.")

Spearman(skill intensity, doc word-count) = 0.142  (p=4.8e-60, n=13097)
low-to-moderate => length is not a strong shortcut for proficiency.


## 5. Document text variance & samples

Quantitative + qualitative diversity of the *generated text* (not just labels):
repeated openings or near-identical structure signal templated, low-variance
generation. Inspired by the old `data/analyze_variance.py` and `data/sample_documents.py`.

In [31]:
import random
from collections import Counter, defaultdict
random.seed(0)
NL = chr(10)

docs_by_type = defaultdict(list)
for d in db.values():
    docs_by_type[d['type']].append(d)

# Opening-line diversity: many docs sharing a first line => templated generation
rows = []
for t, docs in sorted(docs_by_type.items()):
    firsts = [d['text'].strip().split(NL, 1)[0].strip()[:80]
              for d in docs if d.get('text', '').strip()]
    if not firsts:
        continue
    top_line, top_n = Counter(firsts).most_common(1)[0]
    rows.append({'doc_type': t, 'docs': len(docs),
                 'unique_openings': len(set(firsts)),
                 'unique_pct': round(100 * len(set(firsts)) / len(firsts), 1),
                 'most_common_opening': top_line[:45], 'mc_count': top_n})
pd.DataFrame(rows).set_index('doc_type')

,docs,unique_openings,unique_pct,most_common_opening,mc_count
doc_type,,,,,
blog,140,122,87.1,Problem Statement,8
cv,310,234,75.5,Femi Akinlade,3
linkedin,60,60,100.0,Qasim Malik is a product manager at HealthSph,1
project_readme,425,409,96.2,# Real-Time Fraud Detection System,5
recommendation,237,185,78.1,Academic Context,22


In [32]:
# Structural variance: first lines of a few random docs per type
for t, n_lines in [('cv', 12), ('project_readme', 10)]:
    docs = docs_by_type.get(t, [])
    if not docs:
        continue
    print('=' * 70)
    print(f'{t.upper()} - first {n_lines} lines of 3 random docs')
    print('=' * 70)
    for d in random.sample(docs, min(3, len(docs))):
        print(NL + f'--- {d["doc_id"]} (persona {d["persona_id"]}) ---')
        for ln in d['text'].split(NL)[:n_lines]:
            print(ln)

CV - first 12 lines of 3 random docs

--- d_p_197_00 (persona p_197) ---
Samira Mahmoud, a Lead Backend Engineer at NexGen Solutions, based in Bengaluru, India, has spent six years building scalable, high-performance systems for startups and enterprise environments. Her career began as a Junior Backend Developer at TechNova Inc. from 2017 to 2018, where she transitioned from academic theory to real-world application by contributing to a customer portal using Python and PostgreSQL. This role solidified her understanding of REST API design and laid the foundation for her future work in microservices architecture. By 2018, she joined CloudSolutions LLC as a Senior Backend Engineer, where she spearheaded the migration of monolithic applications to a microservices-based architecture. This involved designing and implementing gRPC services in Go, optimizing Redis for session management, and establishing CI/CD pipelines with GitHub Actions. Her work reduced deployment latency by 30% and improv

In [33]:
# Full-text sample, one document per type (truncated for readability)
MAX_CHARS = 700
for t, docs in sorted(docs_by_type.items()):
    d = random.choice(docs)
    print('=' * 70)
    print(f'TYPE: {t.upper()}  (total {len(docs)})  sample {d["doc_id"]} / {d["persona_id"]}')
    print('=' * 70)
    txt = d['text']
    print(txt[:MAX_CHARS] + ('... [truncated]' if len(txt) > MAX_CHARS else ''))
    print()

TYPE: BLOG  (total 140)  sample d_p_225_04 / p_225
**Designing Win-Win Monetization Strategies for Gamers**  

In the gaming industry, the tension between player satisfaction and revenue generation is as old as the free-to-play model itself. As a Senior Product Manager, I’ve seen how monetization strategies can either uplift or alienate players. The key lies in crafting systems that reward both the business and the gamer—a balance achieved through data-driven design, iterative testing, and a deep understanding of player psychology.  

1. **Player-Centric Monetization Frameworks**  
   Monetization shouldn’t feel like a chore. By integrating microtransactions with core gameplay mechanics, we create value that players *want* to pay for. For ex... [truncated]

TYPE: CV  (total 310)  sample d_p_155_00 / p_155
Cedric Blanc  
Email: cedric.blanc@nexacorp.com | Phone: +33 6 12 34 56 78 | Location: Paris, France  

**Professional Experience**  
Senior QA Engineer | NexaCorp (2023–Present)  
Le

### Takeaways
- **Class imbalance is the headline variance issue**: labels 3–4 dominate; level 5 (and the
  near-absent level 1 at persona-skill grain) are thin — weight the loss or rebalance.
- **Flags cluster at high allocated levels**: the judge disputes level-5 claims far more often,
  so any delta-based filter disproportionately thins the rarest class.
- **The `max>=3 or avg>=2` rule drops ~9% of rows**, concentrated on labels 4–5 — a middle ground
  between the loose `|delta|>=2` (~31%) and strict `|delta|>=3` (~7%) cuts.
- Shortcut checks pass (low BoW accuracy, weak length↔level correlation), so labels aren't
  trivially guessable — the model has to actually read the evidence.